# 7 — Reading frequencies off a quantum state

## What you will learn

The last five notebooks built up the vocabulary: superposition, entanglement,
interference, measurement, decoherence, and the combinators that let a program be
inverted and conditioned. This one uses all of it to build the first genuinely
*useful* quantum subroutine — the one every famous quantum algorithm turns out to be
made of.

By the end you will be able to say, precisely:

- what a **discrete Fourier transform** does, and why "it turns a period into a
  position" is the whole idea;
- how the **Quantum Fourier Transform** applies that same map to the $2^n$ amplitudes
  of a register using about $n^2/2$ gates — and why that is *not* a way to compute
  Fourier transforms quickly;
- why the QFT circuit is nothing but Hadamards and controlled phase rotations, by
  reading each gate as writing one **binary digit into a phase**;
- why its output comes out **bit-reversed**, and why that is a labelling issue rather
  than an error;
- how throwing away the smallest rotations (the **approximate QFT**) costs almost
  nothing, and how to measure exactly how little;
- what **phase estimation** is: reading the angle of a unitary's eigenvalue in binary,
  driven by **phase kickback**;
- and how the same answer can be extracted with **one** qubit instead of $t$, by
  measuring early and feeding the bits back classically — the *deferred measurement
  principle* in action.

Everything here is the measuring instrument that Shor's algorithm (notebook 08) reads
its answer through. It is worth going slowly.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

import qsim
from qsim import Circuit, viz
from qsim.algorithms.phase_estimation import (
    phase_estimation,
    semiclassical_phase_estimation,
)
from qsim.algorithms.qft import iqft, qft
from qsim.gates import SWAP, CPhase, H, Phase

np.set_printoptions(precision=3, suppress=True)

## 1. The classical warm-up: a Fourier transform finds a period

Before any quantum mechanics, a five-minute reminder of what a Fourier transform *is*,
because the quantum version is not a different thing — it is the same linear map,
applied to a different list of numbers.

Give the **discrete Fourier transform** (DFT) a list of $N$ numbers $x_0,\dots,x_{N-1}$
and it hands back $N$ numbers

$$y_k \;=\; \sum_{j=0}^{N-1} x_j\, e^{-2\pi i jk/N}$$

Read the sum this way: each input $x_j$ is multiplied by a complex number of magnitude
1 that spins around the unit circle at rate $k$, and the results are added. If the input
wiggles at rate $k$ too, the spinning stays in step with it and the terms pile up. If it
does not, the terms point in all directions and cancel.

So $|y_k|$ answers "how much does this list wiggle $k$ times across its length?" — and
a list that repeats with period $r$ wiggles $N/r$ times, so its energy lands at
$k = N/r$. **The DFT converts a period into a position.** That one sentence is the
reason Shor's algorithm exists.

Here is a sampled cosine that goes round exactly 5 times in 64 samples.

In [ ]:
N = 64
sample = np.arange(N)
signal = np.cos(2 * np.pi * 5 * sample / N)

fig, ax = plt.subplots(figsize=(7, 2.4))
ax.plot(sample, signal, marker="o", markersize=3, linewidth=1, color="crimson")
ax.set_xlabel("sample index j")
ax.set_ylabel("$x_j$")
ax.set_title("a cosine sampled at 64 points — 5 full cycles")
fig.tight_layout()

`np.fft.fft` computes exactly the sum above. Two things about its output layout, since
they trip up everyone the first time:

- The output has the same length as the input, $N = 64$ numbers indexed $k = 0 \dots 63$.
  Index $k$ means "wiggles $k$ times across the whole window".
- A **real** input always produces a symmetric spectrum: index $k$ and index $N-k$ carry
  the same magnitude. That is because a real cosine is the sum of two counter-rotating
  complex waves, one at $+k$ and one at $-k$, and index $N-k$ is how "$-k$" is spelled
  when your indices have to be non-negative. So expect *two* peaks, not one.

In [ ]:
spectrum = np.fft.fft(signal)
peaks = np.argsort(-np.abs(spectrum))[:2]
print("largest two components at k =", sorted(int(k) for k in peaks))

fig, ax = plt.subplots(figsize=(7, 2.4))
ax.bar(np.arange(N), np.abs(spectrum), width=0.8, color="crimson")
ax.set_xlabel("frequency index k")
ax.set_ylabel("|y_k|")
ax.set_title("the spectrum: everything zero except k = 5 and k = 59")
fig.tight_layout()

Sixty-four numbers went in; sixty-two of them came out as (numerically) zero and two
came out large, at exactly the frequency we put in. Nothing quantum has happened yet.
Hold on to the picture — a flat-looking input, a spiky output — because that is exactly
what we are about to build with gates.

## 2. A register's amplitudes are a list of numbers too

An $n$-qubit register holds $2^n$ amplitudes, one per basis state. That is a list of
numbers, so the DFT applies to it, and the **Quantum Fourier Transform** is exactly that
map:

$$\mathrm{QFT}\,\lvert j\rangle \;=\; \frac{1}{\sqrt{2^n}}\sum_{k=0}^{2^n-1}
e^{+2\pi i jk/2^n}\,\lvert k\rangle$$

Two small differences from the formula in section 1, both forced rather than chosen. The
$1/\sqrt{2^n}$ is there because the QFT has to be **unitary** — it is a physical
operation, so it must preserve total probability, and only the symmetric normalization
does that. The sign in the exponent is $+$ rather than $-$ purely by convention; it means
qsim's `qft` matches numpy's *inverse* transform, `np.fft.ifft`, scaled by $\sqrt{N}$.
(Test T11 in
`tests/test_acceptance_t11_t15.py` reconciles the two conventions line by line, and is
worth reading once.)

### Cheating, on purpose, to get a clean picture

To *see* the transform we need a register whose amplitudes trace out a signal. There is
no honest circuit for that here — preparing an arbitrary state is as hard as the problems
we want to solve — so the next cell reaches straight into `qc._psi` and writes the
amplitudes in by hand. **This is simulator-only cheating**, of the same kind that lives
behind `qc.inspect`, and it is worth naming out loud rather than sliding past: in
notebook 08, Shor's algorithm will *earn* its periodic state by computing $a^x \bmod N$
into a second register. Here we are just painting the picture we want to Fourier
transform.

In [ ]:
n = 5
size = 2**n
index = np.arange(size)

# Amplitudes tracing a cosine that goes round 4 times across the register.
amplitudes = np.cos(2 * np.pi * 4 * index / size)
amplitudes = amplitudes / np.linalg.norm(amplitudes)

wave = Circuit(name="wave", seed=0)
reg = wave.register(n, name="q")
# Simulator-only state prep: write the amplitudes straight into the state tensor,
# reshaped to (2,)*n — one axis per qubit, exactly as state.py describes.
wave._psi = amplitudes.reshape((2,) * n).astype(np.complex128)

fig = viz.amplitudes(wave, phase_as_hue=False)

Bar height is $|$amplitude$|$, so the cosine's negative half shows up as height rather
than sign — but the periodicity is plain. Now apply the transform.

In [ ]:
qft(reg)
fig = viz.amplitudes(wave)

peaks = np.argsort(-wave.inspect.probabilities())[:2]
print("the amplitude now lives at basis states", sorted(int(k) for k in peaks))
print("probability of each:", wave.inspect.probabilities()[peaks])

Thirty-two amplitudes collapsed onto two: $\lvert 4\rangle$ and $\lvert 28\rangle$,
half the probability each, and every other basis state is exactly zero. Same picture as
the classical spectrum in section 1 — including the mirror peak at $32 - 4 = 28$, for the
same reason (our amplitudes were real).

**And here is the catch, stated as early as possible.** We can see those two peaks
because `viz.amplitudes` is reading the state vector, which is cheating. Measure the
register instead and you get *one* basis state, 4 or 28, chosen by a coin flip — one
number, out of the 32 the transform computed. The QFT does not let you compute a Fourier
transform faster. It is only useful inside an algorithm arranged so that **one frequency
dominates**, so that the single number you are allowed to see is the one you wanted.

That is not a footnote. It is the shape of every quantum speedup: not "the same
computation, faster", but "a computation whose answer happens to survive measurement".

### The comb — the picture Shor's algorithm actually produces

Change the input from a cosine to a **periodic train of spikes**: equal amplitude on
every basis state $j$ with $j \equiv 0 \pmod r$, zero everywhere else. That is the state
Shor's algorithm ends up with after computing $a^x \bmod N$ and measuring the second
register, and its transform is the picture the whole algorithm is built to look at.

In [ ]:
def comb_state(n: int, period: int) -> Circuit:
    "A register whose amplitudes are equal on multiples of `period` and zero elsewhere."
    amps = np.zeros(2**n)
    amps[::period] = 1.0
    amps /= np.linalg.norm(amps)

    circuit = Circuit(name=f"period {period}", seed=0)
    circuit.register(n, name="q")
    circuit._psi = amps.reshape((2,) * n).astype(np.complex128)
    return circuit


for period in (4, 8):
    circuit = comb_state(5, period)
    qft(circuit.qubits)
    probs = circuit.inspect.probabilities()
    where = sorted(int(k) for k in np.flatnonzero(probs > 1e-9))
    print(f"period r = {period}: peaks at {where}   (spacing 32/r = {32 // period})")
    fig = viz.amplitudes(circuit)

A comb in, a comb out — with the spacing *inverted*. Input period 4 gives peaks every 8;
input period 8 gives peaks every 4. Measure such a register and you land on one of those
peaks, i.e. on a multiple of $2^n/r$; divide by $2^n$ and you have a number close to
$s/r$ for some unknown $s$. Recovering $r$ from that fraction is a classical
number-theory step (continued fractions, notebook 08). **The quantum part of Shor's
algorithm is this plot.**

## 3. How the circuit does it: binary digits written into phases

The QFT circuit is startlingly small — Hadamards and controlled phase rotations, nothing
else. To see why it works, the key is a rewriting of the transform's output that turns a
sum over $2^n$ terms into a product of $n$ independent factors.

### Binary fractions

Write $0.b_1 b_2 b_3\ldots$ for a number in binary *after* the point:

$$0.b_1 b_2 b_3\ldots \;=\; \frac{b_1}{2} + \frac{b_2}{4} + \frac{b_3}{8} + \cdots$$

so $0.011 = 0 + \tfrac14 + \tfrac18 = \tfrac38$, exactly as $0.375$ means
$\tfrac{3}{10}+\tfrac{7}{100}+\tfrac{5}{1000}$ in decimal.

### The product form

With the input written as $\lvert j_1 j_2 \ldots j_n\rangle$ ($j_1$ the most significant
bit, as everywhere in qsim), the transform factorizes:

$$\mathrm{QFT}\lvert j\rangle = \frac{1}{\sqrt{2^n}}
\Big(\lvert 0\rangle + e^{2\pi i\,0.j_n}\lvert 1\rangle\Big)
\Big(\lvert 0\rangle + e^{2\pi i\,0.j_{n-1}j_n}\lvert 1\rangle\Big)\cdots
\Big(\lvert 0\rangle + e^{2\pi i\,0.j_1 j_2\ldots j_n}\lvert 1\rangle\Big)$$

Stare at that for a moment, because it is the whole circuit. Every output qubit is
**unentangled** from the others, and each one carries a binary fraction of the input
number in the phase of its $\lvert 1\rangle$ component — one digit longer than the
qubit before it. (A consequence worth noticing: the QFT of a definite number creates no
entanglement at all. It creates plenty when fed a superposition, which is the only case
anyone cares about.)

Now the gates write themselves:

- **$H$ writes the first digit.** $H\lvert b\rangle = (\lvert 0\rangle + (-1)^b
  \lvert 1\rangle)/\sqrt2$, and $(-1)^b$ is exactly $e^{2\pi i\,0.b}$. One Hadamard
  turns a bit into a one-digit binary fraction sitting in a phase.
- **A controlled phase appends the next digit.** `CPhase(control, target, theta=2π/2^m)`
  multiplies the target's $\lvert 1\rangle$ amplitude by $e^{2\pi i/2^m}$, but only when
  the control is $\lvert 1\rangle$ — which is precisely "append the control's bit at
  binary place $m$".

Here is the three-qubit circuit run gate by gate on the input $\lvert 111\rangle = 7$,
with the state printed in Dirac notation after each step. Watch the first qubit's phase
accumulate digits.

In [ ]:
walk = Circuit(name="walk", seed=0)
w = walk.register(3, name="q")
w.encode(7)  # |111> — every control is 1, so every rotation below actually fires
print("start                   ", walk.inspect.ket())

H(w[0])
print("H on q0                 ", walk.inspect.ket())
CPhase(w[1], w[0], theta=2 * np.pi / 2**2)
print("+ digit from q1 (1/4)   ", walk.inspect.ket())
CPhase(w[2], w[0], theta=2 * np.pi / 2**3)
print("+ digit from q2 (1/8)   ", walk.inspect.ket())

H(w[1])
CPhase(w[2], w[1], theta=2 * np.pi / 2**2)
print("q1 done                 ", walk.inspect.ket())

H(w[2])
print("q2 done (bit-reversed)  ", walk.inspect.ket())

### Why the output comes out backwards

Look at which factor each qubit ended up holding. Qubit 0 was processed **first**, so it
collected rotations from every qubit after it and ended up carrying the **longest**
fraction $0.j_1j_2j_3$ — which the product formula says belongs on the *last* output
qubit. Qubit 2 was processed last, collected nothing, and carries the shortest fraction
$0.j_3$, which belongs first.

The circuit produces the correct transform **in reversed qubit order**. Nothing is wrong
with that state; it is the right answer written back-to-front. A network of SWAPs fixes
it, and `qft(reg)` applies it by default. Pass `swap=False` to see the reversal for
yourself.

In [ ]:
SWAP(w[0], w[2])
print("after the swap network  ", walk.inspect.ket())

target = np.exp(2j * np.pi * 7 * np.arange(8) / 8) / np.sqrt(8)
print("\nmatches the formula exactly:",
      np.abs(walk.inspect.state_vector() - target).max() < 1e-12)

And the same claim as a single comparison: `swap=False` differs from `swap=True` by a
reversal of the register, which — since the axes of the state tensor *are* the qubits —
is nothing but a transpose of the array with its axis list reversed.

In [ ]:
unswapped = Circuit(seed=0)
u = unswapped.register(3)
u.encode(3)
qft(u, swap=False)

swapped = Circuit(seed=0)
s = swapped.register(3)
s.encode(3)
qft(s)

# Reverse the axis order of the (2,2,2) state tensor, then flatten: re-reading the
# same amplitudes with the register held the other way round.
reordered = np.transpose(unswapped.inspect.state_tensor(), [2, 1, 0]).reshape(-1)
print("reversing the axes turns one into the other:",
      np.abs(reordered - swapped.inspect.state_vector()).max() < 1e-12)

### What it costs, and that the inverse is free

One $H$ per qubit and one controlled rotation per *pair* of qubits: $n + n(n-1)/2$
gates, about $n^2/2$, to transform $2^n$ amplitudes. A classical FFT of the same $2^n$
numbers needs about $2^n \cdot n$ arithmetic operations. For $n = 20$: 210 gates against
twenty million multiply-adds.

The inverse transform needs no new circuit at all. `iqft` is built as `qft.adjoint()` —
the recorded body replayed backwards with every gate inverted — using exactly the block
algebra from notebook 04. It is the direction you will actually use, since *reading* a
frequency out of a register is an inverse transform.

In [ ]:
counts = Circuit(seed=0)
qft(counts.register(8))
print("8-qubit QFT gate counts:", counts.gate_counts())
print("n(n-1)/2 =", 8 * 7 // 2)

roundtrip = Circuit(seed=0)
r = roundtrip.register(4)
r.encode(9)
qft(r)
iqft(r)
print("\nqft then iqft returns |9>:", roundtrip.inspect.ket())
print("blocks recorded on the tape:", roundtrip.block_counts())

## 4. The approximate QFT: most of those rotations were doing nothing

The smallest rotation in an $n$-qubit QFT is by $2\pi/2^n$. At $n = 12$ that is a
fifteenth of a degree; at $n = 20$ it is a millionth of a turn. No physical device
applies a millionth of a turn accurately, so if the algorithm genuinely needed those
angles it would be unbuildable.

It does not. `qft(reg, approx=m)` skips every rotation finer than $1/2^m$ of a turn, and
the error it introduces scales like $t^2 2^{-m}$ — so a handful of distinct angles
suffices no matter how large the register. Here is that curve, measured: fidelity of the
truncated transform against the exact one, on a random 12-qubit state.

(**Fidelity** was introduced in notebook 03: $|\langle\phi|\psi\rangle|^2$, the
probability that a measurement asking "are you this state?" says yes. 1 means identical,
0 means perfectly distinguishable.)

In [ ]:
digits = 12
rng = np.random.default_rng(11)
random_amps = rng.normal(size=(2,) * digits) + 1j * rng.normal(size=(2,) * digits)
random_amps /= np.linalg.norm(random_amps)


def transformed(approx: int | None) -> Circuit:
    "The same random 12-qubit state, transformed with rotations truncated at `approx`."
    circuit = Circuit(digits, seed=0)
    circuit._psi = random_amps.astype(np.complex128)
    qft(circuit.qubits, approx=approx)
    return circuit


exact = transformed(None).inspect.state_vector()
levels = range(1, 13)
fidelities = [transformed(m).inspect.fidelity(exact) for m in levels]

for m, f in zip(levels, fidelities, strict=True):
    print(f"approx = {m:2d}   fidelity {f:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(list(levels), 1 - np.array(fidelities), marker="o", color="crimson")
ax.axhline(1e-3, color="#17797c", linestyle="--", linewidth=1)
ax.text(1.2, 1.3e-3, "1 - fidelity = 0.001", color="#17797c")
ax.set_yscale("log")
ax.set_xlabel("truncation level m — rotations finer than $1/2^m$ are dropped")
ax.set_ylabel("1 - fidelity  (log scale)")
ax.set_title("what the approximate QFT gives up, on 12 qubits")
fig.tight_layout()

Straight line on a log axis: each extra binary place of rotation cuts the error by about
half again, exactly as $t^2 2^{-m}$ predicts. By $m = 8$ the fidelity is above $0.9997$.

How much circuit does that buy back? At twelve qubits, honestly, not much — and it is
worth printing the real numbers rather than gesturing at the asymptotics.

In [ ]:
def rotation_count(n: int, approx: int | None = None) -> int:
    "Controlled rotations in an n-qubit QFT, exact or truncated at `approx`."
    return sum(
        1
        for j in range(n)
        for k in range(j + 1, n)
        if approx is None or k - j + 1 <= approx
    )


truncated = Circuit(seed=0)
qft(truncated.register(12), approx=8)
print("12-qubit QFT, exact:      ", rotation_count(12), "controlled rotations")
print("12-qubit QFT, m = 8:      ", truncated.gate_counts()["CPhase"],
      "controlled rotations")
print("                fidelity: ", f"{fidelities[7]:.6f}")

print()
for width in (12, 50, 200):
    print(f"n = {width:3d}:  exact {rotation_count(width):6d}"
          f"    truncated at m = 8 {rotation_count(width, 8):6d}")

At $n = 12$, truncating removes 10 rotations out of 66 — a rounding error. The point is
what happens as the register grows: the exact circuit needs $n(n-1)/2$ rotations,
**quadratic** in $n$, while a fixed truncation level needs about $m\,n$, **linear**. By
$n = 200$ that is roughly 20,000 gates against 1,400, and the fidelity cost does not grow
with $n$ — only with $t^2 2^{-m}$, which the choice of $m$ controls directly.

So the lesson is not "approximation is acceptable here". It is that the exact circuit
carries a quadratic tail of gates that are never earning their keep, and that a quantum
algorithm's cost is worth interrogating gate by gate rather than read off an asymptotic
formula.

## 5. Phase estimation: reading an eigenvalue as an angle

Now the payoff. The QFT is a tool; **phase estimation** is what the tool is for, and
almost every quantum algorithm with an exponential speedup is phase estimation in a
costume.

### Every eigenvalue of a unitary is an angle

A unitary $U$ preserves lengths — that is what makes it a physically possible operation.
So if $U\lvert v\rangle = \lambda \lvert v\rangle$, the lengths of $\lvert v\rangle$ and
$\lambda\lvert v\rangle$ must agree, forcing $|\lambda| = 1$. Every eigenvalue of every
unitary sits on the unit circle, and can therefore be written

$$\lambda = e^{2\pi i \varphi}, \qquad \varphi \in [0, 1)$$

There is exactly **one real number per eigenvector** to find, and it is a fraction of a
full turn. Phase estimation finds $\varphi$ to $t$ binary digits.

An **eigenvector** (notebook 01's linear algebra, no new physics) is a state $U$ leaves
pointing in the same direction, changing only by that overall factor. Here is
$\varphi = 3/8$ drawn where it lives.

In [ ]:
phi_example = 3 / 8

fig, ax = plt.subplots(figsize=(3.6, 3.6))
angles = np.linspace(0, 2 * np.pi, 400)
ax.plot(np.cos(angles), np.sin(angles), color="#8a8f98", linewidth=1)
ax.plot([0, np.cos(2 * np.pi * phi_example)], [0, np.sin(2 * np.pi * phi_example)],
        color="crimson", linewidth=2)
ax.plot([np.cos(2 * np.pi * phi_example)], [np.sin(2 * np.pi * phi_example)],
        "o", color="crimson")
ax.annotate(r"$e^{2\pi i \varphi},\ \varphi = 3/8$", xy=(-0.68, 0.78),
            color="crimson")
ax.set_aspect("equal")
ax.set_xlim(-1.35, 1.35)
ax.set_ylim(-1.25, 1.25)
ax.axhline(0, color="#8a8f98", linewidth=0.6)
ax.axvline(0, color="#8a8f98", linewidth=0.6)
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title("an eigenvalue is a point on the unit circle")
fig.tight_layout()

### Phase kickback

The whole algorithm rests on one small surprise. Take a control qubit in
$(\lvert 0\rangle + \lvert 1\rangle)/\sqrt2$, a target holding an eigenstate
$\lvert v\rangle$, and apply a **controlled**-$U$:

$$\tfrac{1}{\sqrt2}(\lvert 0\rangle + \lvert 1\rangle)\lvert v\rangle
\;\longrightarrow\; \tfrac{1}{\sqrt2}\big(\lvert 0\rangle\lvert v\rangle
+ \lvert 1\rangle\, U\lvert v\rangle\big)
= \tfrac{1}{\sqrt2}\big(\lvert 0\rangle + e^{2\pi i\varphi}\lvert 1\rangle\big)
\otimes \lvert v\rangle$$

The target came out **completely unchanged** — an eigenstate is by definition what $U$
does not move — and the phase landed on the qubit that merely *watched*. That is
**phase kickback**: the gate appears to act on the target, and its entire visible effect
appears on the control.

Notice why the effect is visible at all. A phase multiplying the *whole* state is
unobservable; a phase on *one branch of a superposition* is not. The control had to be
in superposition for there to be anywhere to put the answer, which is why the circuit
starts with a Hadamard on every output qubit.

Watch it happen on the Bloch sphere. The control starts at $\lvert +\rangle$, on the
equator pointing along $x$; after the kickback it has rotated around the equator by
exactly $2\pi\varphi$.

In [ ]:
def phase_unitary(phi: float) -> qsim.Block:
    "A one-qubit block with eigenvalue exp(2*pi*i*phi) on |1> and 1 on |0>."

    @qsim.gate
    def u(register: qsim.Register) -> None:
        Phase(register[0], theta=2 * np.pi * phi)

    return u


kick = Circuit(name="kickback", seed=0)
control = kick.alloc("control")
target = kick.register(1, name="v")
target.encode(1)  # |1> is the eigenvector whose eigenvalue we want

H(control)
print("control after H: bloch =", kick.inspect.bloch_vector(control))

phase_unitary(3 / 8).controlled(control)(target)
print("after controlled-U:      ", kick.inspect.bloch_vector(control))
print("expected (cos 2pi*3/8, sin 2pi*3/8, 0) =",
      (np.cos(2 * np.pi * 3 / 8), np.sin(2 * np.pi * 3 / 8), 0.0))
print("the target is still |1>, unentangled:",
      kick.inspect.is_product(target), kick.inspect.marginal(target))

fig = viz.bloch(control)

One qubit, one application of $U$, and the angle is *there* — but only as a phase, which
no measurement of that qubit can read. To turn it into bits we need many such qubits,
each holding the same angle multiplied by a different power of two, and then an inverse
QFT to convert that phase pattern into a number.

That is the whole circuit, and `phase_estimation` is eight lines:

1. $H$ on every qubit of the output register;
2. for each output qubit, apply controlled-$U$ a number of times equal to its place
   value — $2^{t-1}$ times for the most significant, down to once for the least;
3. `iqft` on the output register;
4. measure.

After step 2 the output register holds precisely $\mathrm{QFT}\lvert y\rangle$ for
$y = 2^t\varphi$ — the product form from section 3, with the binary fractions of $y$
sitting in the phases. Step 3 undoes the transform and leaves $\lvert y\rangle$ itself.

### The exactly representable case

$\varphi = 3/8$ is $0.011$ in binary — three digits, no remainder. A three-qubit output
register should therefore return $011 = 3$ with certainty.

In [ ]:
qc = Circuit(name="pe-exact", seed=0)
eigen = qc.register(1, name="v")
out = qc.register(3, name="out")
eigen.encode(1)

phase_estimation(phase_unitary(3 / 8), eigen, out)

# marginal() gives the distribution over just the output register, MSB first.
# (probabilities()[3] would be a different and wrong number: the probability of the
# whole four-qubit state |0011>.)
outcomes = qc.inspect.marginal(out)
print("P(out = 011) =", outcomes[0b011])
print("full distribution over out:", outcomes)
print("\nmeasured:", format(qc.measure_all(out), "03b"))

Probability 1, to floating-point precision. Every one of the $2^3$ interfering paths
through the circuit arrives at $\lvert 011\rangle$ in phase and at every wrong answer out
of phase — the same cancellation that made $H$ twice give $\lvert 0\rangle$ in notebook
04, running eight ways at once.

### The normal case: a phase that has no exact binary expansion

Take $\varphi = 0.3$. In binary that is $0.0100110011\ldots$, repeating forever, so no
finite register can hold it. With $t = 5$ digits the best available answer is
$10/32 = 0.3125$, and the distribution piles up around it instead of collapsing onto it.

In [ ]:
spread = Circuit(name="pe-spread", seed=0)
spread_eigen = spread.register(1, name="v")
spread_out = spread.register(5, name="out")
spread_eigen.encode(1)

phase_estimation(phase_unitary(0.3), spread_eigen, spread_out)
distribution = spread.inspect.marginal(spread_out)

for y in np.argsort(-distribution)[:4]:
    print(f"y = {y:2d}  ->  phi ~ {y / 32:.4f}   probability {distribution[y]:.4f}")

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.bar(np.arange(32), distribution, width=0.8, color="crimson")
ax.axvline(32 * 0.3, color="#17797c", linestyle="--", linewidth=1)
ax.text(32 * 0.3 + 0.5, distribution.max() * 0.8, r"true $2^t\varphi = 9.6$",
        color="#17797c")
ax.set_xlabel("measured value y")
ax.set_ylabel("probability")
ax.set_title(r"phase estimation of $\varphi = 0.3$ with 5 digits")
fig.tight_layout()

Peaked but spread: 57% on $y = 10$, 25% on $y = 9$, and a thin tail. The two most likely
answers are the two representable phases straddling $0.3$, which is the best any $t$-bit
answer could do — and adding qubits narrows the peak rather than moving it.

This is the normal situation, and it is why Shor's algorithm needs a classical
post-processing step: what comes out is *close to* the phase, and turning a nearby
fraction back into the exact one it approximates is a job for continued fractions.

## 6. The semiclassical version: $t$ digits from one qubit

The coherent circuit above keeps $t$ output qubits alive in superposition until the very
end. On real hardware, qubits are the scarce resource. Griffiths and Niu noticed in 1996
that you do not need them.

Look again at what the inverse QFT does when every qubit is about to be measured anyway.
The qubit measured *first* needs no correction at all; every later one needs a rotation
whose angle depends only on bits **already measured**. And bits already measured are
ordinary classical numbers, which an ordinary Python variable can hold.

So: keep **one** qubit. Round after round, put it in superposition, kick a phase onto it,
rotate it by an angle computed from the digits collected so far, undo the superposition,
measure, and reset. The classical feedback replaces the entanglement.

That should feel familiar. It is the same move as Bob's correction in teleportation
(notebook 03): "classical communication" inside a quantum protocol is never anything more
exotic than an `if` reading a bit.

In [ ]:
one_qubit = Circuit(name="semiclassical", seed=0)
one_target = one_qubit.register(1, name="v")
one_target.encode(1)

# Track the peak qubit count with a tape hook (notebook 04) rather than claiming it.
peak: list[int] = []
handle = one_qubit.on_op(lambda op, circuit: peak.append(circuit.n_qubits))
answer = semiclassical_phase_estimation(phase_unitary(3 / 8), one_target, 3)
handle.remove()

print("answer:", format(answer, "03b"), "=", answer)
print("most qubits alive at any moment:", max(peak))
print("the coherent version needed:    ", 1 + 3)

The phase qubit is borrowed with `qc.ancilla(1)`, so on the way out the simulator
*verifies* it came back to $\lvert 0\rangle$ and unentangled — the honest bookkeeping for
a qubit that has been measured and reset three times over.

### Why measuring early is allowed

It looks as though measuring partway through must destroy something. Quantum mechanics is
famously unforgiving about when you measure, and notebook 04 spent a whole section on a
single scratch qubit wrecking an interference pattern.

The rescue is the **deferred measurement principle**: a measurement followed by a
classically-controlled gate is *provably equivalent* to a quantum-controlled gate
followed by a measurement at the end. Moving a measurement later never changes any
outcome distribution — so moving it earlier does not either, provided nothing downstream
needs the branches it separated to interfere. Here nothing does: those branches only ever
decide a rotation angle.

Provable is good; measured is better. Below, 2000 seeded runs of the one-qubit version
against the coherent circuit's exact distribution.

In [ ]:
shots = 2000
counts: Counter[int] = Counter()
for seed in range(shots):
    run = Circuit(seed=seed)
    run_target = run.register(1, name="v")
    run_target.encode(1)
    counts[semiclassical_phase_estimation(phase_unitary(0.3), run_target, 5)] += 1

sampled = np.array([counts[y] / shots for y in range(32)])
tvd = 0.5 * np.abs(sampled - distribution).sum()
print(f"total-variation distance over {shots} shots: {tvd:.4f}")

fig, ax = plt.subplots(figsize=(7, 3))
grid = np.arange(32)
ax.bar(grid - 0.2, distribution, width=0.4, color="crimson", label="coherent (exact)")
ax.bar(grid + 0.2, sampled, width=0.4, color="#17797c",
       label=f"semiclassical ({shots} runs)")
ax.set_xlabel("measured value y")
ax.set_ylabel("probability")
ax.set_title("one reused qubit reproduces the five-qubit circuit")
ax.legend()
fig.tight_layout()

The two histograms sit on top of each other, and the total-variation distance —
$\tfrac12\sum_y |p_y - q_y|$, the largest difference in probability the two distributions
assign to any event — is at the level sampling noise alone would produce from 2000 draws.
Acceptance test T15 pins this down at 500 shots with a tolerance of 0.05.

Two different circuits, using a different number of qubits and measuring at different
times, produce the same distribution. That is a real theorem about quantum mechanics,
checked rather than recited.

## What you now know

- The **discrete Fourier transform** turns a period into a position: a list that repeats
  with period $r$ has its energy at frequency index $N/r$. Everything else follows from
  that one sentence.
- The **QFT** is that same map applied to a register's $2^n$ amplitudes, built from
  $n$ Hadamards and $n(n-1)/2$ controlled phase rotations — about $n^2/2$ gates where a
  classical FFT needs $2^n n$ operations.
- **It is not a fast Fourier transform.** The output is amplitudes, and measuring returns
  one basis state. The QFT is useful only inside an algorithm arranged so one frequency
  dominates. Every quantum speedup has this shape: not the same computation faster, but a
  computation whose answer survives measurement.
- The circuit works because the transform **factorizes**: each output qubit carries one
  binary fraction of the input in its phase. $H$ writes the first digit; each controlled
  rotation appends the next.
- The output is **bit-reversed** — the qubit processed first ends up holding the last
  factor — and the SWAP network relabels it. Reversing a register is a transpose of the
  state tensor's axes, which is what `swap=False` lets you see.
- `iqft` is `qft.adjoint()`. The block algebra from notebook 04 means the inverse
  transform costs no new circuit.
- The **approximate QFT** (`approx=m`) drops rotations finer than $1/2^m$, with error
  $\sim t^2 2^{-m}$. On 12 qubits, $m = 8$ keeps fidelity above $0.9997$ — most of those
  rotations were never earning their keep.
- **Phase estimation** reads $\varphi$ from $U$'s eigenvalue $e^{2\pi i \varphi}$ in
  binary, using $H$'s, repeated controlled-$U$'s, and one `iqft`. It is the universal
  quantum measuring instrument.
- **Phase kickback** is the mechanism: applied to an eigenstate, controlled-$U$ leaves
  the target untouched and deposits the eigenvalue's phase on the *control*. The target
  is a catalyst.
- The target **need not be an eigenstate**. Feed a superposition and the circuit runs on
  every eigenvector at once, returning one phase at random, entangled with the
  eigenvector that produced it. That is not a fallback — it is exactly how Shor's
  algorithm works.
- The **semiclassical** version gets the same distribution from one reused qubit plus
  classical feedback, which the **deferred measurement principle** guarantees and T15
  confirms.

## Next: notebook 08 — but first, arithmetic

Phase estimation needs a unitary to estimate. Shor's algorithm supplies "multiply by $a$
modulo $N$", and that raises a problem this notebook has quietly sidestepped: the
unitaries used here were single `Phase` gates, chosen because their eigenvalues were
known in advance. A real one has to be **built**.

Building it means doing arithmetic reversibly — every gate invertible, no bits discarded,
because a discarded bit is a record, and a record destroys the interference the whole
algorithm depends on (notebook 04, section 4). Addition, multiplication and modular
reduction all have to be compiled out of Toffoli gates and undone again afterwards with
Bennett's trick.

That reversible adding machine is the last ingredient, and it is the largest single piece
of engineering in this project. Once it exists, Shor's algorithm is this notebook's
`phase_estimation` called on it — and the comb from section 2 appears for real, earned
rather than painted.